# 02 — Interpolate USCRN to a 2-degree CONUS grid

For each day and variable, fit:
- IDW (baseline)
- Ordinary Kriging with **spherical**, **exponential**, **gaussian** variograms

Output: one NetCDF per variable with dims `(time, lat, lon, model)`.

Runtime: ~365 days x 5 vars x 3 OK fits ~= 5500 fits at <0.5 s each
(~30 min on a laptop). IDW is essentially free.


In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm
from pykrige.ok import OrdinaryKriging

warnings.filterwarnings('ignore')

ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(ROOT))
from utils import target_grid, ensure_dirs

ensure_dirs(ROOT)
TIDY = ROOT / 'data' / 'uscrn' / 'uscrn_daily_tidy.csv'
GRIDS = ROOT / 'grids'


In [2]:
lon_c, lat_c = target_grid()
LONG, LATG = np.meshgrid(lon_c, lat_c)
print('grid shape:', LATG.shape)

df = pd.read_csv(TIDY, parse_dates=['date'])
print('rows:', len(df), '| stations:', df['station_id'].nunique())


grid shape: (13, 30)
rows: 47961 | stations: 132


## Interpolation kernels

In [3]:
def idw(x, y, v, xg, yg, power=2, eps=1e-9):
    xg = xg.ravel(); yg = yg.ravel()
    d = np.sqrt((xg[:, None] - x[None, :]) ** 2 + (yg[:, None] - y[None, :]) ** 2)
    w = 1.0 / (d ** power + eps)
    return ((w * v[None, :]).sum(axis=1) / w.sum(axis=1)).reshape(LONG.shape)

def ok_grid(x, y, v, xg, yg, model):
    OK = OrdinaryKriging(
        x, y, v, variogram_model=model,
        verbose=False, enable_plotting=False,
        nlags=12, weight=True,
    )
    z, _ = OK.execute('grid', xg, yg)
    return np.asarray(z)


## Daily loop

In [4]:
VARS = ['precip', 'rh', 't_air', 't_soil', 'sm']
MODELS = ['idw', 'spherical', 'exponential', 'gaussian']
MIN_STATIONS = 20

dates = pd.date_range(df['date'].min(), df['date'].max(), freq='D')

def grid_one_day(sub, model):
    x = sub['lon'].values; y = sub['lat'].values; v = sub['val'].values
    if model == 'idw':
        return idw(x, y, v, LONG, LATG)
    try:
        return ok_grid(x, y, v, lon_c, lat_c, model)
    except Exception:
        return np.full(LONG.shape, np.nan)

for var in VARS:
    arr = np.full((len(dates), len(lat_c), len(lon_c), len(MODELS)), np.nan, dtype=np.float32)
    for ti, d in enumerate(tqdm(dates, desc=var)):
        sub = df.loc[df['date'] == d, ['lon', 'lat', var]].dropna()
        if len(sub) < MIN_STATIONS:
            continue
        sub = sub.rename(columns={var: 'val'})
        for mi, m in enumerate(MODELS):
            arr[ti, :, :, mi] = grid_one_day(sub, m)
    ds = xr.Dataset(
        {var: (('time', 'lat', 'lon', 'model'), arr)},
        coords={'time': dates, 'lat': lat_c, 'lon': lon_c, 'model': MODELS},
        attrs={'description': f'USCRN-derived {var} on 2-deg CONUS grid'},
    )
    out = GRIDS / f'uscrn_grid_2deg_{var}.nc'
    ds.to_netcdf(out)
    print('wrote', out)


precip:   0%|          | 0/366 [00:00<?, ?it/s]

wrote /home/askeladd/Downloads/gnr640_project/grids/uscrn_grid_2deg_precip.nc


rh:   0%|          | 0/366 [00:00<?, ?it/s]

wrote /home/askeladd/Downloads/gnr640_project/grids/uscrn_grid_2deg_rh.nc


t_air:   0%|          | 0/366 [00:00<?, ?it/s]

wrote /home/askeladd/Downloads/gnr640_project/grids/uscrn_grid_2deg_t_air.nc


t_soil:   0%|          | 0/366 [00:00<?, ?it/s]

wrote /home/askeladd/Downloads/gnr640_project/grids/uscrn_grid_2deg_t_soil.nc


sm:   0%|          | 0/366 [00:00<?, ?it/s]

wrote /home/askeladd/Downloads/gnr640_project/grids/uscrn_grid_2deg_sm.nc
